In [7]:
import pandas as pd
import numpy as np
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision.datasets import CIFAR10

In [29]:
# load datasets and dataloaders

from torch.utils.data import DataLoader
import torchvision.transforms as transforms

# there is two transformation we are going to do will scall the image first then normalize it
# image => scale(0,1) => normalize =>(-1,1)


transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5) , (0.5, 0.5, 0.5))
])

trainset = CIFAR10(root="./datasets",train=True,download=True,transform=transform)
testset = CIFAR10(root="./datasets",train=False,download=True,transform=transform)


trainloader = DataLoader(trainset,batch_size=32,shuffle=True)
testloader = DataLoader(testset,batch_size=32)

In [37]:
# build our CNN

class CNN(nn.Module):
    
    
    def __init__(self):
        super(CNN,self).__init__()
    
    
        self.conv_layer = nn.Sequential(
            nn.Conv2d(3,32 ,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
    
            nn.Conv2d(32,64 ,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
    
            nn.Conv2d(64,128 ,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2)
        )
    
        self.FC_Layer = nn.Sequential(
            nn.Linear(4*4*128,256),
            nn.ReLU(),
            nn.Linear(256,10)
        )

    def forward(self,x):
        x = self.conv_layer(x)
        x = x.view(x.size(0),-1)   #flattning
        x = self.FC_Layer(x)
        return x

In [38]:
# assign CNN 

model = CNN()

criterian = nn.CrossEntropyLoss()
optimizers = optim.Adam(model.parameters())

In [ ]:
# training data

epochs = 10

for epoch in range(epochs):
    epoch_training_loss = 0.00

    for images, labels in trainloader:
        optimizers.zero_grad()
        output = model.forward(images)
        loss = criterian(output,labels)
        loss.backward()
        optimizers.step()

        epoch_training_loss += loss.item()

    print(f"epoch{epoch+1}/{epochs} & loss = {epoch_training_loss/len(trainloader)}")

In [ ]:
# model evaluation

correct_label = 0
total_labes = 0
model.eval()

with torch.no_grad():
  for image,labels in testloader:
    output = model.forward(image)
    loss = criterian(output,labels)

    predicted = torch.max(output.data,1)[1]
    total_labes += labels.size(0)
    correct_label += (predicted == labels).sum().item()
    
print(f"accuracy = {100*correct_label/total_labes}")
    




In [ ]:

from PIL import Image
from IPython.display import display
import ipywidgets as widgets

uploader = widgets.FileUpload(
    accept=".jpg,.jpeg,.png",
    multiple=False
)

display(uploader)



In [ ]:

import io
import torch
import matplotlib.pyplot as plt
from PIL import Image

class_names = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

# Get uploaded image
uploaded = uploader.value

if isinstance(uploaded, dict):  # Newer ipywidgets
    file_info = next(iter(uploaded.values()))
    image_bytes = file_info["content"]
else:  # Older ipywidgets
    file_info = uploaded[0]
    image_bytes = file_info["content"]

# Open and prepare image
image = Image.open(io.BytesIO(bytes(image_bytes))).convert("RGB")

# CIFAR-10 images are 32 x 32 pixels
image = image.resize((32, 32))

# Use the SAME transform used for your test dataset
input_tensor = transform(image).unsqueeze(0)

model.eval()

with torch.no_grad():
    output = model(input_tensor)
    probabilities = torch.softmax(output, dim=1)[0]
    predicted_class = probabilities.argmax().item()

# Display original image
plt.imshow(image)
plt.axis("off")
plt.show()

print("Predicted class:", class_names[predicted_class])
print(f"Confidence: {probabilities[predicted_class].item() * 100:.2f}%")